# 00 - Data Understanding & Exploratory Analysis (Phase 1)
**Portfolio Project**: E-Commerce Customer Intelligence & Recommendation Platform  
**Dataset**: Olist Brazilian E-Commerce Dataset (9 CSVs in `data/raw/`)

### Objectives:
1. **Time coverage & monthly volume**: Identify marketplace growth, seasonality, and truncations.
2. **Order status breakdown**: Confirm valid statuses for completed order revenue.
3. **Customer repeat purchases**: Contrast `customer_id` (order session token) with `customer_unique_id` (individual human).
4. **Review text coverage & star ratings**: Quantify proportion of reviews with written Portuguese feedback vs star rating only.
5. **Logistics & late deliveries**: Quantify late delivery rate and inter-state fulfillment distribution across Brazil.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure repo root is on sys.path when running from notebooks/ or subdirectories
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import DATA_RAW, ANALYSIS_START, ANALYSIS_END, VALID_STATUSES

print(f"Data raw path: {DATA_RAW}")
print(f"Analysis window: {ANALYSIS_START} to {ANALYSIS_END}")
print(f"Valid statuses: {VALID_STATUSES}")


Data raw path: C:\Users\KRISHAN\Desktop\Fun\Projects\Ecommerce_Customer_Intelligence\data\raw
Analysis window: 2017-01-01 to 2018-08-31
Valid statuses: ['delivered']


## 1. Load Datasets
Load the raw CSVs into DataFrames.


In [2]:
df_orders = pd.read_csv(DATA_RAW / "olist_orders_dataset.csv")
df_customers = pd.read_csv(DATA_RAW / "olist_customers_dataset.csv")
df_items = pd.read_csv(DATA_RAW / "olist_order_items_dataset.csv")
df_reviews = pd.read_csv(DATA_RAW / "olist_order_reviews_dataset.csv")
df_sellers = pd.read_csv(DATA_RAW / "olist_sellers_dataset.csv")

print(f"Orders:      {len(df_orders):,} rows")
print(f"Customers:   {len(df_customers):,} rows")
print(f"Order Items: {len(df_items):,} rows")
print(f"Reviews:     {len(df_reviews):,} rows")
print(f"Sellers:     {len(df_sellers):,} rows")


Orders:      99,441 rows
Customers:   99,441 rows
Order Items: 112,650 rows
Reviews:     99,224 rows
Sellers:     3,095 rows


## 2. Time Coverage and Orders Per Month
Examine the purchase timestamp span to detect pilot ramp-up and dataset truncation.


In [3]:
df_orders["order_purchase_timestamp"] = pd.to_datetime(df_orders["order_purchase_timestamp"])
min_date = df_orders["order_purchase_timestamp"].min()
max_date = df_orders["order_purchase_timestamp"].max()

print(f"Earliest order purchase: {min_date}")
print(f"Latest order purchase:   {max_date}")

df_orders["year_month"] = df_orders["order_purchase_timestamp"].dt.to_period("M")
monthly_orders = df_orders["year_month"].value_counts().sort_index().to_frame(name="order_count")
monthly_orders["share_pct"] = (monthly_orders["order_count"] / len(df_orders) * 100).round(2)
display(monthly_orders)


Earliest order purchase: 2016-09-04 21:15:19
Latest order purchase:   2018-10-17 17:30:18


,order_count,share_pct
year_month,,
2016-09,4,0.00
2016-10,324,0.33
2016-12,1,0.00
2017-01,800,0.80
2017-02,1780,1.79
2017-03,2682,2.70
2017-04,2404,2.42
2017-05,3700,3.72
2017-06,3245,3.26


## 3. Order Status Breakdown
Identify the proportion of orders across statuses (delivered, shipped, canceled, etc.).


In [4]:
status_dist = df_orders["order_status"].value_counts().to_frame(name="count")
status_dist["pct"] = (status_dist["count"] / len(df_orders) * 100).round(2)
display(status_dist)


,count,pct
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


## 4. Customer Identity & Repeat-Buyer Share
Contrast `customer_id` (order-level token) with `customer_unique_id` (actual user) to compute repeat purchase rate.


In [5]:
merged_cust = df_orders.merge(df_customers, on="customer_id", how="inner")

cust_counts_all = merged_cust.groupby("customer_unique_id")["order_id"].nunique()
repeat_all = (cust_counts_all > 1).sum()
total_all = len(cust_counts_all)

delivered_cust = merged_cust[merged_cust["order_status"] == "delivered"]
cust_counts_deliv = delivered_cust.groupby("customer_unique_id")["order_id"].nunique()
repeat_deliv = (cust_counts_deliv > 1).sum()
total_deliv = len(cust_counts_deliv)

print(f"Total unique customers: {total_all:,}")
print(f"Repeat customers (All statuses): {repeat_all:,} ({repeat_all / total_all * 100:.2f}%)")
print(f"Single-purchase customers: {total_all - repeat_all:,} ({(total_all - repeat_all) / total_all * 100:.2f}%)")
print(f"Repeat customers (Delivered only): {repeat_deliv:,} ({repeat_deliv / total_deliv * 100:.2f}%)")


Total unique customers: 96,096
Repeat customers (All statuses): 2,997 (3.12%)
Single-purchase customers: 93,099 (96.88%)
Repeat customers (Delivered only): 2,801 (3.00%)


## 5. Review Text Coverage & Rating Distribution
Analyze star scores and the availability of Portuguese review titles and comments.


In [6]:
n_reviews = len(df_reviews)
n_titles = df_reviews["review_comment_title"].notna().sum()
n_messages = df_reviews["review_comment_message"].notna().sum()
n_either = (df_reviews["review_comment_title"].notna() | df_reviews["review_comment_message"].notna()).sum()

print(f"Total reviews: {n_reviews:,}")
print(f"Has review title:    {n_titles:,} ({n_titles / n_reviews * 100:.2f}%)")
print(f"Has review message:  {n_messages:,} ({n_messages / n_reviews * 100:.2f}%)")
print(f"Has either comment:  {n_either:,} ({n_either / n_reviews * 100:.2f}%)")

score_dist = df_reviews["review_score"].value_counts().sort_index().to_frame(name="count")
score_dist["pct"] = (score_dist["count"] / n_reviews * 100).round(2)
display(score_dist)


Total reviews: 99,224
Has review title:    11,568 (11.66%)
Has review message:  40,977 (41.30%)
Has either comment:  42,706 (43.04%)


,count,pct
review_score,,
1,11424,11.51
2,3151,3.18
3,8179,8.24
4,19142,19.29
5,57328,57.78


## 6. Delivery Delays & Inter-State Fulfillment Mix
Examine shipping performance vs estimated delivery dates and inter-state logistics across Brazil.


In [7]:
deliv = df_orders[df_orders["order_status"] == "delivered"].copy()
deliv["order_delivered_customer_date"] = pd.to_datetime(deliv["order_delivered_customer_date"])
deliv["order_estimated_delivery_date"] = pd.to_datetime(deliv["order_estimated_delivery_date"])

valid_deliv = deliv.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"]).copy()
valid_deliv["is_late"] = valid_deliv["order_delivered_customer_date"] > valid_deliv["order_estimated_delivery_date"]
late_count = valid_deliv["is_late"].sum()
total_valid = len(valid_deliv)

print(f"Delivered orders evaluated: {total_valid:,}")
print(f"Late orders: {late_count:,} ({late_count / total_valid * 100:.2f}%)")

# State match analysis
items_geo = df_items.merge(
    df_customers[["customer_id", "customer_state"]].merge(df_orders[["order_id", "customer_id"]], on="customer_id"),
    on="order_id"
).merge(df_sellers[["seller_id", "seller_state"]], on="seller_id")

same_state_count = (items_geo["customer_state"] == items_geo["seller_state"]).sum()
total_items = len(items_geo)

print(f"Total order items evaluated: {total_items:,}")
print(f"Same-state deliveries:  {same_state_count:,} ({same_state_count / total_items * 100:.2f}%)")
print(f"Inter-state deliveries: {total_items - same_state_count:,} ({(total_items - same_state_count) / total_items * 100:.2f}%)")


Delivered orders evaluated: 96,470
Late orders: 7,826 (8.11%)
Total order items evaluated: 112,650
Same-state deliveries:  40,756 (36.18%)
Inter-state deliveries: 71,894 (63.82%)
